# Module 1 Homework (2026)

Notebook for [homework1.md](../homework1.md).

Submit: https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw01


## Setup


In [1]:
# Install + import libraries (SSL patch before HTTPS clients).
%pip install yfinance pandas lxml beautifulsoup4 requests truststore curl_cffi

import truststore
truststore.inject_into_ssl()

import requests
import pandas as pd
import numpy as np
import yfinance as yf
from curl_cffi import requests as curl_requests
from io import StringIO
from datetime import date

# yfinance uses curl_cffi; on some Windows setups OS CA store fails for curl.
# Shared session with verify=False keeps Yahoo Finance downloads working locally.
yf_session = curl_requests.Session(impersonate="chrome", verify=False)



Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Question 1: S&P 500 Stocks Added to the Index

**Which year had the highest number of additions (starting from 2020)?**


In [2]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    )
}
response = requests.get(url, headers=headers)
response.raise_for_status()

tables = pd.read_html(StringIO(response.text))
sp500 = tables[0].copy()
sp500["Date added"] = pd.to_datetime(sp500["Date added"], errors="coerce")
sp500["year_added"] = sp500["Date added"].dt.year

adds_by_year = sp500.dropna(subset=["year_added"]).groupby("year_added").size()
adds_2020_plus = adds_by_year[adds_by_year.index >= 2020]
q1_year = int(adds_2020_plus.idxmax())
q1_count = int(adds_2020_plus.max())

cutoff = pd.Timestamp(date.today()) - pd.DateOffset(years=20)
n_over_20y = int((sp500["Date added"].notna() & (sp500["Date added"] <= cutoff)).sum())
n_over_20y_incl_missing = int(
    ((sp500["Date added"].isna()) | (sp500["Date added"] <= cutoff)).sum()
)

print("Additions by year (>=2020):")
print(adds_2020_plus.sort_index())
print(f"\nQ1 answer: {q1_year} ({q1_count} additions)")
print(f"Additional: stocks with Date added >20y ago (dated only): {n_over_20y}")
print(f"Additional: including undated original members: {n_over_20y_incl_missing}")



Additions by year (>=2020):
year_added
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
2026    13
dtype: int64

Q1 answer: 2025 (18 additions)
Additional: stocks with Date added >20y ago (dated only): 224
Additional: including undated original members: 224


## Question 2: Indexes YTD (as of 21 August 2026)

**How many indexes (out of 10) have better YTD returns than the US (S&P 500)?**


In [3]:
indexes = {
    "United States": "^GSPC",
    "China": "000001.SS",
    "Hong Kong": "^HSI",
    "Australia": "^AXJO",
    "India": "^NSEI",
    "Canada": "^GSPTSE",
    "Germany": "^GDAXI",
    "United Kingdom": "^FTSE",
    "Japan": "^N225",
    "Mexico": "^MXX",
    "Brazil": "^BVSP",
}

start_ytd = "2026-01-01"
end_inclusive = pd.Timestamp("2026-08-21")
# yfinance end is exclusive
end_download = "2026-08-22"

rows = []
for name, ticker in indexes.items():
    hist = yf.Ticker(ticker, session=yf_session).history(
        start=start_ytd, end=end_download, auto_adjust=False
    )
    if hist.empty:
        print(f"WARN empty: {name} {ticker}")
        continue
    hist = hist.copy()
    if getattr(hist.index, "tz", None) is not None:
        hist.index = hist.index.tz_localize(None)
    hist = hist[hist.index <= end_inclusive]
    close_start = hist["Close"].iloc[0]
    close_end = hist["Close"].iloc[-1]
    ret = close_end / close_start - 1
    rows.append(
        {
            "name": name,
            "ticker": ticker,
            "start_date": hist.index[0].date(),
            "end_date": hist.index[-1].date(),
            "close_start": close_start,
            "close_end": close_end,
            "ytd_return": ret,
        }
    )

ytd = pd.DataFrame(rows).set_index("name")
us_ytd = ytd.loc["United States", "ytd_return"]
others = ytd.drop(index="United States")
q2_count = int((others["ytd_return"] > us_ytd).sum())

print(ytd[["ticker", "start_date", "end_date", "ytd_return"]].sort_values("ytd_return", ascending=False))
print(f"\nUS YTD: {us_ytd:.4%}")
print(f"Q2 answer: {q2_count} indexes beat S&P 500 YTD")


def period_returns(years):
    start = (end_inclusive - pd.DateOffset(years=years)).strftime("%Y-%m-%d")
    out = {}
    for name, ticker in indexes.items():
        hist = yf.Ticker(ticker, session=yf_session).history(
            start=start, end=end_download, auto_adjust=False
        )
        if hist.empty:
            continue
        hist = hist.copy()
        if getattr(hist.index, "tz", None) is not None:
            hist.index = hist.index.tz_localize(None)
        hist = hist[hist.index <= end_inclusive]
        out[name] = hist["Close"].iloc[-1] / hist["Close"].iloc[0] - 1
    s = pd.Series(out)
    us = s["United States"]
    beat = int((s.drop(labels=["United States"]) > us).sum())
    return beat, us, s


for yrs in (3, 5, 10):
    beat, us, _ = period_returns(yrs)
    print(f"Additional {yrs}y: {beat}/10 beat US (US={us:.4%})")



                   ticker  start_date    end_date  ytd_return
name                                                         
Japan               ^N225  2026-01-05  2026-08-21    0.273641
Canada            ^GSPTSE  2026-01-02  2026-08-21    0.148566
United States       ^GSPC  2026-01-02  2026-08-21    0.118962
United Kingdom      ^FTSE  2026-01-02  2026-08-21    0.086975
Brazil              ^BVSP  2026-01-02  2026-08-21    0.065361
Germany            ^GDAXI  2026-01-02  2026-08-21    0.065088
Australia           ^AXJO  2026-01-02  2026-08-21    0.037936
Mexico               ^MXX  2026-01-02  2026-08-21    0.024755
Hong Kong            ^HSI  2026-01-02  2026-08-21   -0.012492
China           000001.SS  2026-01-05  2026-08-21   -0.029382
India               ^NSEI  2026-01-01  2026-08-21   -0.072459

US YTD: 11.8962%
Q2 answer: 2 indexes beat S&P 500 YTD
Additional 3y: 2/10 beat US (US=74.4266%)
Additional 5y: 2/10 beat US (US=71.3209%)
Additional 10y: 1/10 beat US (US=251.6095%)


## Question 3: S&P 500 Market Corrections — median drawdown %

Correction = drawdown of at least 5% from most recent all-time high.


In [4]:
spx = yf.Ticker("^GSPC", session=yf_session).history(start="1950-01-01", auto_adjust=False)
spx = spx.copy()
if getattr(spx.index, "tz", None) is not None:
    spx.index = spx.index.tz_localize(None)
close = spx["Close"]

# All-time high days: close exceeds all previous closes
prev_max = close.cummax().shift(1)
is_ath = close > prev_max
is_ath.iloc[0] = True
ath_dates = close.index[is_ath]
ath_prices = close[is_ath]

corrections = []
for i in range(len(ath_dates) - 1):
    t0, t1 = ath_dates[i], ath_dates[i + 1]
    high = float(ath_prices.iloc[i])
    # Segment from ATH day through day before next ATH
    segment = close.loc[t0:t1].iloc[:-1]
    if segment.empty:
        continue
    trough_date = segment.idxmin()
    low = float(segment.min())
    dd = (high - low) / high * 100
    duration = (trough_date - t0).days
    corrections.append(
        {
            "ath_date": t0.date(),
            "trough_date": trough_date.date(),
            "next_ath": t1.date(),
            "high": high,
            "low": low,
            "drawdown_pct": dd,
            "duration_days": duration,
        }
    )

corr_df = pd.DataFrame(corrections)
sig = corr_df[corr_df["drawdown_pct"] >= 5].copy()

q3_median_dd = float(sig["drawdown_pct"].median())
print("Significant corrections (>=5%):", len(sig))
print("Drawdown percentiles:")
print(sig["drawdown_pct"].quantile([0.25, 0.5, 0.75]))
print("Duration percentiles:")
print(sig["duration_days"].quantile([0.25, 0.5, 0.75]))
print(f"\nQ3 answer (median drawdown %): {q3_median_dd:.4f}")

print("\nTop 10 by drawdown:")
print(
    sig.nlargest(10, "drawdown_pct")[
        ["ath_date", "trough_date", "drawdown_pct", "duration_days"]
    ].to_string(index=False)
)



Significant corrections (>=5%): 74
Drawdown percentiles:
0.25     6.234677
0.50     7.986358
0.75    14.019826
Name: drawdown_pct, dtype: float64
Duration percentiles:
0.25    22.00
0.50    40.50
0.75    86.25
Name: duration_days, dtype: float64

Q3 answer (median drawdown %): 7.9864

Top 10 by drawdown:
  ath_date trough_date  drawdown_pct  duration_days
2007-10-09  2009-03-09     56.775388            517
2000-03-24  2002-10-09     49.146948            929
1973-01-11  1974-10-03     48.203593            630
1968-11-29  1970-05-26     36.061641            543
2020-02-19  2020-03-23     33.924960             33
1987-08-25  1987-12-04     33.509515            101
1961-12-12  1962-06-26     27.973568            196
1980-11-28  1982-08-12     27.113582            622
2022-01-03  2022-10-12     25.425097            282
1966-02-09  1966-10-07     22.177335            240


## Question 4: AMZN Earnings Surprise — median 2-day return after positive surprises


In [11]:
ticker = "AMZN"
amzn = yf.Ticker(ticker, session=yf_session)
# Default get_earnings_dates() ~25 rows from 2020-10-29 (homework spec)
earnings = amzn.get_earnings_dates()
earnings = earnings.copy()
# Drop future / incomplete rows (NaN Reported EPS / Surprise%)
earnings = earnings.dropna(subset=["Reported EPS", "Surprise(%)"])
print("Earnings rows (reported):", len(earnings))
print(earnings.head())
print("...")
print(earnings.tail())

prices = amzn.history(period="max", auto_adjust=False)
prices = prices.copy()
if getattr(prices.index, "tz", None) is not None:
    prices.index = prices.index.tz_localize(None)
closes = prices["Close"]

# 2-day return for Day2 = i: Close[i+1]/Close[i-1]-1
ret_2d = closes.shift(-1) / closes.shift(1) - 1
ret_2d.name = "ret_2d"

earn = earnings.copy()
earn.index = pd.to_datetime(earn.index)
if getattr(earn.index, "tz", None) is not None:
    earn.index = earn.index.tz_localize(None)
earn.index = earn.index.normalize()
earn = earn[~earn.index.duplicated(keep="first")]
earn["ret_2d"] = ret_2d.reindex(earn.index)

# If earnings date is not a trading day, map to next available trading day
missing = earn["ret_2d"].isna()
if missing.any():
    for dt in earn.index[missing]:
        later = closes.index[closes.index >= dt]
        if len(later):
            earn.loc[dt, "ret_2d"] = ret_2d.get(later[0], np.nan)

positive = earn[earn["Surprise(%)"] > 0].dropna(subset=["ret_2d"])
q4_median = float(positive["ret_2d"].median())
print(f"\nPositive surprises with returns: {len(positive)}")
print(f"Q4 answer (median 2-day return): {q4_median:.6f} ({q4_median:.4%})")

corr = earn[["ret_2d", "Surprise(%)"]].dropna().corr()
print("\nCorrelation (all surprises):")
print(corr)
print("corr(ret_2d, Surprise%):", float(corr.loc["ret_2d", "Surprise(%)"]))



Earnings rows (reported): 24
                           EPS Estimate  Reported EPS  Surprise(%)
Earnings Date                                                     
2026-07-30 16:00:00-04:00          1.83          5.75       215.02
2026-04-29 16:00:00-04:00          1.64          2.78        69.02
2026-02-05 16:00:00-05:00          1.95          1.95         0.22
2025-10-30 16:00:00-04:00          1.56          1.95        25.20
2025-07-31 16:00:00-04:00          1.32          1.68        27.19
...
                           EPS Estimate  Reported EPS  Surprise(%)
Earnings Date                                                     
2021-10-28 16:00:00-04:00          0.44          0.31       -31.21
2021-07-29 16:00:00-04:00          0.61          0.76        23.06
2021-04-29 16:00:00-04:00          0.47          0.79        67.30
2021-02-02 16:00:00-05:00          0.35          0.70       100.10
2020-10-29 16:00:00-04:00          0.38          0.62        64.25

Positive surprises with retu

In [12]:
# EPS estimates: Yahoo Finance consensus analyst EPS, column "EPS Estimate"
# from Ticker.get_earnings_dates() (earnings calendar, not Ticker.info / financials).
# Public page: https://finance.yahoo.com/calendar/earnings?symbol=AMZN
# Surprise(%) is Yahoo's published beat/miss; abs surprise = Reported EPS - EPS Estimate.
# price_growth_2d uses the Q4 window: Close[d3] / Close[d1] - 1, d2 = announcement session.

q4_events = earn.copy()
q4_events["earnings_date"] = q4_events.index.normalize()
q4_events["ticker"] = ticker
q4_events["reported_eps"] = q4_events["Reported EPS"]
q4_events["eps_estimate"] = q4_events["EPS Estimate"]
q4_events["eps_surprise"] = q4_events["reported_eps"] - q4_events["eps_estimate"]
q4_events["eps_surprise_pct"] = q4_events["Surprise(%)"]
q4_events["price_growth_2d"] = q4_events["ret_2d"]

q4_table = (
    q4_events[
        [
            "ticker",
            "earnings_date",
            "reported_eps",
            "eps_estimate",
            "eps_surprise",
            "eps_surprise_pct",
            "price_growth_2d",
        ]
    ]
    .sort_values("eps_surprise_pct", ascending=False)
    .reset_index(drop=True)
)

print(q4_table.to_string(index=False))


ticker earnings_date  reported_eps  eps_estimate  eps_surprise  eps_surprise_pct  price_growth_2d
  AMZN    2022-02-03          1.39          0.18          1.21            657.12         0.046656
  AMZN    2026-07-30          5.75          1.83          3.92            215.02         0.198235
  AMZN    2021-02-02          0.70          0.35          0.35            100.10        -0.009079
  AMZN    2023-08-03          0.65          0.34          0.31             91.19         0.088605
  AMZN    2026-04-29          2.78          1.64          1.14             69.02         0.020639
  AMZN    2021-04-29          0.79          0.47          0.32             67.30         0.002579
  AMZN    2020-10-29          0.62          0.38          0.24             64.25        -0.040038
  AMZN    2023-10-26          0.94          0.58          0.36             62.22         0.052311
  AMZN    2023-04-27          0.31          0.21          0.10             44.29         0.004477
  AMZN    2022-10-27

## Question 5 (optional): Capstone project idea

Повний список компаній і проєктів (категорії A / B / C / D, з тікерами): [q5-ai-infra-capstone/INFRA-CAPSTONE.md](q5-ai-infra-capstone/INFRA-CAPSTONE.md)

Копія в портфоліо: [../../projects/head_of/02-ai-infra-supply-chain/INFRA-CAPSTONE.md](../../projects/head_of/02-ai-infra-supply-chain/INFRA-CAPSTONE.md)

Зріз моделей: [q5-ai-infra-capstone/openrouter_value leaders.csv](q5-ai-infra-capstone/openrouter_value%20leaders.csv)

**Теза.** Куди йде наступний долар AI-інфраструктури — не прогноз «чи NVIDIA виросте за 5 днів».

**Чотири книги**
- **A.** 50 глобальних публічних компаній без материкового Китаю (чипи, заводи, HBM, мережа, стійки, охолодження, хмара, дата-центри).
- **B.** 20 китайських імен заліза й хмари окремим контуром (частина без біржового тікера, наприклад Huawei).
- **C.** Енергетичні компанії з тікерами плюс великі проєкти живлення кампусів.
- **D.** Компанії за топовими моделями OpenRouter: MiniMax `0100.HK`, GLM/Z.AI `2513.HK`, Qwen/Alibaba, Tencent, Xiaomi, DeepSeek, Kimi. Приватні лабораторії без тікера теж тримаємо в списку.

Рішення: який шар зараз вузьке місце (GPU / HBM / електрика / лабораторія моделей) і чи sleeve invest / hold / underweight.

Метрики (питання 6): [q6-ai-metrics/Q6-METRICS.md](q6-ai-metrics/Q6-METRICS.md)



## Question 6 (optional): Additional metrics to explore

Які метрики рахувати і як вони лягають на ланцюг постачання (платник → технологія → підрядник): [q6-ai-metrics/Q6-METRICS.md](q6-ai-metrics/Q6-METRICS.md)

Коротко:
1. **CapEx платників** — Microsoft (робочий орієнтир близько 20 млрд доларів, завжди звіряти з 10-Q), Amazon, Google, Meta, Oracle, Apple, NVIDIA.
2. **Матриця підрядників** — кому доходять ці гроші (`NVDA`, `TSM`, `000660.KS`, `VRT`, `CEG`, `ETR` тощо).
3. **Енергетика** — оголошені ГВт і статус проєктів з категорії C.
4. **Макро** — `^SOX`, FRED `IPB53122S`, `VIXCLS`, `T10Y2Y`, `FEDFUNDS`.
5. **Звіти** — сюрприз EPS (рецепт питання 4), Wikipedia GICS, Yahoo `info`.

Китайську книгу не змішуємо з глобальною. Деталі й формули — у файлі вище.



## Answers summary (submission form)


In [6]:
print("=== HW1 ANSWERS ===")
print(f"Q1 year with most S&P500 additions (>=2020): {q1_year}")
print(f"Q2 # indexes with better YTD than US (of 10): {q2_count}")
print(f"Q3 median correction drawdown %: {q3_median_dd}")
print(f"Q4 median 2-day return after positive earnings surprise: {q4_median}")
print("Q5/Q6: see markdown cells above")
print("Submit: https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw01")



=== HW1 ANSWERS ===
Q1 year with most S&P500 additions (>=2020): 2025
Q2 # indexes with better YTD than US (of 10): 2
Q3 median correction drawdown %: 7.986357561745869
Q4 median 2-day return after positive earnings surprise: 0.0035280649406906894
Q5/Q6: see markdown cells above
Submit: https://courses.datatalks.club/sma-zoomcamp-2026/homework/hw01
